In [1]:
select * from dlk.rbsdata2_mta_history 
limit 10 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 2, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 60 fields>

In [4]:
Create or Replace Table stg.A0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, 4a9151b2-504a-4fd1-93ec-85d201fb74a4, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
select min(ConversationStartTime),  max(ConversationStartTime) from stg.A0_genesys_derived_data limit 10 

StatementMeta(, 4a9151b2-504a-4fd1-93ec-85d201fb74a4, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [7]:
select count(*) from stg.A0_genesys_derived_data limit 10 

StatementMeta(, 4a9151b2-504a-4fd1-93ec-85d201fb74a4, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
select  count(distinct ConversationId), count(distinct CustomerPhoneNumber)
from    stg.A0_genesys_derived_data 
Where   QueueName in ('INBOUND_VehicleChangeAddress',
'INBOUND_VehicleChangeOther',
'INBOUND_VehicleChange_Perm',
'INBOUND_VehicleChange_Temp',
'INBOUND_VehicleUpdateLicence',
'INBOUND_Vehicle_Add_Driver')
and     originatingDirection = 'inbound' 


StatementMeta(, 4a9151b2-504a-4fd1-93ec-85d201fb74a4, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [7]:
select  count(*)
from    ods.EventStream
Where	EventDate between '2026-05-01' and '2026-07-31'  /* Marie saw the three month count than 1 month count -- ugghhh */ 
and     EventDescription in ('inbound - INBOUND_SALES_MOTOR')

StatementMeta(, 1ecae669-511e-49ee-b323-6bd8fa99e93b, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
select  count(distinct ConversationId), count(distinct CustomerPhoneNumber)
from    stg.A0_genesys_derived_data 
Where   QueueName in ('INBOUND_SALES_MOTOR')
and     originatingDirection = 'inbound'

StatementMeta(, 1ecae669-511e-49ee-b323-6bd8fa99e93b, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [8]:
Create or Replace Table stg.AbandonedMTACalls as 
select distinct ConversationId
from 
(select	a.ConversationId, min(conversationStartTime) as conversationStartTime, 
            sum(agentAnswered) as agentAnswered, sum(alertNoAnswer) alertNoAnswer, sum(abandoned) abandoned, 
            sum(totalAcdWaitDuration) CallWaitTime, sum(totalAgentAlertDuration) CallRingTime, sum(totalAgentHoldDuration) CallHoldTime, sum(totalAgentTalkDuration) CallSpokenTime
from	stg.A0_genesys_derived_data	a
where	a.queueName in ('INBOUND_VehicleChangeAddress',
'INBOUND_VehicleChangeOther',
'INBOUND_VehicleChange_Perm',
'INBOUND_VehicleChange_Temp',
'INBOUND_VehicleUpdateLicence',
'INBOUND_Vehicle_Add_Driver')
Group by ConversationId) x 
where abandoned = 1 and CallWaitTime > 0 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
select  count(distinct a.ConversationId), count(distinct CustomerPhoneNumber)
from    stg.A0_genesys_derived_data a, stg.AbandonedMTACalls b 
Where   a.ConversationId = b.ConversationId 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [12]:
select  EventDescription , count(distinct SourcePolicyReference)
from    ods.EventStream 
Where   EventSourceId = 3 
and     EventDescription like 'Permanent%' 
and     PolicyTypeGroup = 'Motor' 
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
Group by  EventDescription
Order by 2 desc 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 13, Finished, Available, Finished, False)

<Spark SQL result set with 47 rows and 2 fields>

In [13]:
Create Table stg.MotorMTAPolicies as 
select  distinct SourcePolicyReference
from    ods.EventStream Where EventSourceId = 3 and EventDescription like 'Permanent%' and PolicyTypeGroup = 'Motor' 
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'


StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
select count(*) from stg.MotorMTAPolicies 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [17]:
select  count(distinct x.SourcePolicyReference) 
From 
        (select  SourcePolicyReference, max(EventDateTime) EventDateTime
        from    ods.EventStream 
        Where   EventSourceId = 3 and EventDescription like 'Permanent MTA%' and PolicyTypeGroup = 'Motor' 
        and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
        Group by SourcePolicyReference) x,
        (select  SourcePolicyReference, max(EventDateTime) EventDateTime
        from    ods.EventStream 
        Where EventSourceId = 3 and EventDescription like 'Duplicate Cert%' and PolicyTypeGroup = 'Motor' 
        and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
        Group by SourcePolicyReference) y 
Where   x.SourcePolicyReference = y.SourcePolicyReference
and     x.EventDateTime < y.EventDateTime

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [23]:
select  count(distinct  b.SourcePolicyReference ) from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a, stg.MotorMTAPolicies  b 
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and PolicyTypeGroup = 'Motor' and a.PolicyCode = b.SourcePolicyReference 
and Campaign = 'DAY 1'


StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 24, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [24]:
select  count(distinct  b.SourcePolicyReference ) from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a, stg.MotorMTAPolicies  b 
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and PolicyTypeGroup = 'Motor' and a.PolicyCode = b.SourcePolicyReference 
and Campaign = 'DAY 20'


StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [25]:
select  count(distinct  b.SourcePolicyReference ) from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a, stg.MotorMTAPolicies  b, dlk.MyChill_NewUploadDocumentEvents  c 
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and     PolicyTypeGroup = 'Motor' and a.PolicyCode = b.SourcePolicyReference 
and     Campaign = 'DAY 1'
and     a.PolicyCode = c.PolicyCode
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
;


StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [27]:
select  isAccepted, count(distinct b.PolicyCode)
from    (
            select  b.SourcePolicyReference 
            from    edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a, stg.MotorMTAPolicies  b, dlk.MyChill_NewUploadDocumentEvents  c 
            Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
            and     PolicyTypeGroup = 'Motor' and a.PolicyCode = b.SourcePolicyReference 
            and     Campaign = 'DAY 1'
            and     a.PolicyCode = c.PolicyCode
            and     PolicyTypeGroup = 'Motor'
            and     `Timestamp` > '2026-07-01 00:00:00.000' 
            Group by b.SourcePolicyReference 
        ) a, 
        dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.SourcePolicyReference = b.PolicyCode
and     `Timestamp` > '2026-07-01 00:00:00.000' 
Group by isAccepted

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 28, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [28]:
select  count(distinct x.SourcePolicyReference) 
From 
        (select  SourcePolicyReference, max(EventDateTime) EventDateTime
        from    ods.EventStream 
        Where   EventSourceId = 3 and EventDescription like 'Permanent MTA%' and PolicyTypeGroup = 'Motor' 
        and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
        Group by SourcePolicyReference) x,
        (select  SourcePolicyReference, max(EventDateTime) EventDateTime
        from    ods.EventStream 
        Where EventSourceId = 3 and EventDescription like 'Insurer led%' and PolicyTypeGroup = 'Motor' 
        and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
        Group by SourcePolicyReference) y 
Where   x.SourcePolicyReference = y.SourcePolicyReference
and     x.EventDateTime < y.EventDateTime

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 29, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
/* docs asked for  */
Create or replace table tmp.MotorEscalated20days as 
select  
		distinct PolicyCode 
from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and PolicyType = 'New Business' 
and PolicyTypeGroup = 'Motor' 
and case 
        when Sign_Prop_Status				= 'O' then 1 
        when Sign_Stat_Of_Fact_Status		= 'O' then 1 
        when Comp_Fin_Agree_Status			= 'O' then 1 
        when Comp_DDM_Status				= 'O' then 1 
        when Proof_Of_NCB_Status			= 'O' then 1 
        when Trans_Proof_Of_NCB_Status		= 'O' then 1 
        when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
        when Prop_Driv_Lic_Status			= 'O' then 1 
        when Nam_Driv_Lic_Status			= 'O' then 1 
        when Gap_In_Cov_Ltr_Status			= 'O' then 1 
        when `2ndCar_Cert_Status`			= 'O' then 1 
        when Doc_Ltr_Status					= 'O' then 1 
        when Engineers_Rpt_Status			= 'O' then 1 
        when NCT_Status						= 'O' then 1 
        when Veh_Lic_Cert_Status			= 'O' then 1 
        when Irish_Reg_Status				= 'O' then 1 
        when Main_Driver_Dec_Status			= 'O' then 1 
        when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
        when Comp_Car_Exper_Status			= 'O' then 1 
        when DD_Conf_Ltr_Status				= 'O' then 1 
        when Comp_Fin_Mand_Status			= 'O' then 1 
        when Val_For_Spec_Item_Status		= 'O' then 1 
        when Dri_Lic_Num_Status				= 'O' then 1 
        when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
        when Orig_Cert_Status				= 'O' then 1 
        when Cancel_Req_Status				= 'O' then 1 
        when PPS_Num_Status					= 'O' then 1 
        when Afford_State_Status			= 'O' then 1 
        when Identification_Status			= 'O' then 1 
        when Digital_Journey_Status			= 'O' then 1 
        when Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and Campaign = 'DAY 20'

StatementMeta(, 4d87fc4e-e23b-4af3-a839-bf0f01d1fb46, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
select count(*) from 
    (select	distinct SourcePolicyReference from ods.eventstream 
    where	EventSourceId = 3 
    and		EventDate between '2026-07-01' and '2026-07-31' 
    and		PolicyTypeGroup = 'Motor'
    and		upper(EventDescription) like '%CHASE%' 
    and		upper(EventDescription) like '%FINAL%' ) x ,
    tmp.MotorEscalated20days y 
Where     SourcePolicyReference = PolicyCode



StatementMeta(, 4d87fc4e-e23b-4af3-a839-bf0f01d1fb46, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
Create or Replace Table stg.A0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

In [ ]:
Create or Replace Table stg.DocumentRequest as 
select	a.ConversationId, CustomerPhoneNumber, conversationStartTime
from	stg.A0_genesys_derived_data	a
where	a.queueName in ('INBOUND_Documents_Out')
Group by a.ConversationId, CustomerPhoneNumber,  conversationStartTime


StatementMeta(, df205802-cbc2-437e-b3a7-93f9b9447436, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [30]:
select count(distinct CustomerPhoneNumber) from stg.DocumentRequest

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
Create  or replace Table stg.DocRequestCustomers as 
select  distinct SCV_Customer_Key, SourceSystemReference, conversationStartTime
from    ods.scv_customer_key a,
        (select  RapierCustomerId , conversationStartTime
        from     stg.DocumentRequest a, pii.customer_data_assembled b 
        Where   a.CustomerPhoneNumber = b.CustomerPhone 
        Group by RapierCustomerId, conversationStartTime) b         
where   a.ChillSourceCustomerKey = b.RapierCustomerId
and     SourceSystemId = 1 

StatementMeta(, df205802-cbc2-437e-b3a7-93f9b9447436, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
Create or Replace Table stg.DocRequestPolicies as 
select  distinct SourcePolicyReference, conversationStartTime
from    ods.scveadmlinkfinal a, stg.DocRequestCustomers b 
Where   a.SCV_Customer_Key = b.SCV_Customer_Key 
and     a.SourceCustomerReference = b.SourceSystemReference
and     SourceSystemId = 1 


StatementMeta(, df205802-cbc2-437e-b3a7-93f9b9447436, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
/* Motor	Doc Request	D.V1
*/
select  EventDescription, count(*), count(distinct b.SourcePolicyReference) 
from    stg.DocRequestPolicies a, ods.EventStream b 
Where   a.SourcePolicyReference = b.SourcePolicyReference and b.EventSourceId = 3 
and     b.EventDateTime > conversationStartTime
and     b.EventDateTime < dateadd(day,2,conversationStartTime)
and     b.PolicyTypeGroup = 'Motor' 
and     b.EventDescription like 'Duplicate Certificate%'
and     b.EventDescription = 'Duplicate Certificate - Document Transmitted - Certificate'
Group by EventDescription
Order by 2 desc 

StatementMeta(, df205802-cbc2-437e-b3a7-93f9b9447436, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [25]:
select count(distinct b.SourcePolicyReference) from stg.DocRequestPolicies a, ods.EventStream b 
Where a.SourcePolicyReference = b.SourcePolicyReference and b.EventSourceId = 3 and b.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and b.PolicyTypeGroup = 'Motor' 


StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [27]:
select count(*), count(distinct b.SourcePolicyReference) from stg.DocRequestPolicies a, ods.EventStream b 
Where a.SourcePolicyReference = b.SourcePolicyReference and b.EventSourceId = 3 and b.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and b.PolicyTypeGroup = 'Motor' 
and EventDescription like 'Renewal Offer -%' 


StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [4]:
select EventDescription, count(*), count(distinct b.SourcePolicyReference) from ods.EventStream b 
Where b.EventSourceId = 3 and b.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and b.PolicyTypeGroup = 'Motor' 
and     EventDescription like '%Final%'
-- and     EventDescription like '%scalation%'
Group by EventDescription
Order by 2 desc 

StatementMeta(, c706355c-0dce-45c7-93e0-06703e372434, 5, Finished, Available, Finished, False)

<Spark SQL result set with 13 rows and 3 fields>

In [2]:
select EventDescription, count(*), count(distinct b.SourcePolicyReference) from ods.EventStream b 
Where b.EventSourceId = 3 and b.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and b.PolicyTypeGroup = 'Motor' 
and     EventDescription like 'Insurer %'
Group by EventDescription
Order by 2 desc 

StatementMeta(, c706355c-0dce-45c7-93e0-06703e372434, 3, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 3 fields>

In [41]:
select EventDescription, count(*), count(distinct b.SourcePolicyReference) from stg.DocRequestPolicies a, ods.EventStream b 
Where a.SourcePolicyReference = b.SourcePolicyReference and b.EventSourceId = 3 and b.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and b.PolicyTypeGroup = 'Motor' 
Group by EventDescription
Order by 2 desc 

StatementMeta(, 4283529a-0475-4d9f-8e7b-caff869f4a7a, 42, Finished, Available, Finished, False)

<Spark SQL result set with 649 rows and 3 fields>

In [ ]:

select  PolicyCode, LyPolicyRenewDateAdj as RenewalDate, LYReportingSaleDate, TyPolicyCode , TYReportingSaleDate, TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium


from    edw.tbl_fact_policy_renewals    a 
Where   a. LyPolicyTypeGroup = 'Motor' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
-- and     PolicyOfferNum = 1 
Group by   PolicyCode, LyPolicyRenewDateAdj, LYReportingSaleDate,  TyPolicyCode  , TYReportingSaleDate,  TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium

In [4]:
SELECT  *
from    dlk.EXT_XtremePushResults_Policy
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like '%Motor%' 
and		campaign_name like '%Renewals%' 
and     interaction_type = 'sent' 
and     MessageType in ('EMAIL', 'SMS') 
limit 100 
;

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 5, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 17 fields>

In [11]:
SELECT  count(distinct user_id)
from    dlk.EXT_XtremePushResults_Policy a, (select PolicyCode from stg.R0_MotorPoliciesEligibleForRenewals where coalesce(PolicyRetNum,0) <> 1) b
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like '%Motor%' 
and     a.user_id = b.PolicyCode
and		campaign_name = 'Motor - This Year Lapsed'
and     interaction_type = 'sent' 
and     MessageType in ('EMAIL', 'SMS') 
limit 100 
;

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [21]:
select count(distinct PolicyCode) from stg.R0_MotorPoliciesEligibleForRenewals 
where RenewalDate < coalesce(TyReportingSaleDate , dateadd(day,1,RenewalDate))

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [24]:
select count(distinct PolicyCode) from stg.R0_MotorPoliciesEligibleForRenewals 
where datediff(day,RenewalDate,TyReportingSaleDate) between -3 and 3 

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [17]:
select * from stg.R0_MotorPoliciesEligibleForRenewals limit 100 

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 18, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 11 fields>

In [22]:
SELECT  count(distinct a.user_id)
from    dlk.EXT_XtremePushResults_Policy a, stg.R0_MotorPoliciesEligibleForRenewals b 
where	campaign_name like '%Motor%' 
and     a.user_id = b.PolicyCode
and     SourceFileDate > RenewalDate
and		campaign_name = 'Motor - This Year Lapsed'
and     interaction_type = 'sent' 
and     MessageType in ('EMAIL', 'SMS') 
limit 100 
;

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
SELECT  count(distinct a.PolicyCode)
FROM    edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and     PolicyType = 'Renewals' 
and     PolicyTypeGroup = 'Motor' 
and 
        case 
            when Sign_Prop_Status				= 'O' then 1 
            when Sign_Stat_Of_Fact_Status		= 'O' then 1 
            when Comp_Fin_Agree_Status			= 'O' then 1 
            when Comp_DDM_Status				= 'O' then 1 
            when Proof_Of_NCB_Status			= 'O' then 1 
            when Trans_Proof_Of_NCB_Status		= 'O' then 1 
            when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
            when Prop_Driv_Lic_Status			= 'O' then 1 
            when Nam_Driv_Lic_Status			= 'O' then 1 
            when Gap_In_Cov_Ltr_Status			= 'O' then 1 
            when `2ndCar_Cert_Status`			= 'O' then 1 
            when Doc_Ltr_Status					= 'O' then 1 
            when Engineers_Rpt_Status			= 'O' then 1 
            when NCT_Status						= 'O' then 1 
            when Veh_Lic_Cert_Status			= 'O' then 1 
            when Irish_Reg_Status				= 'O' then 1 
            when Main_Driver_Dec_Status			= 'O' then 1 
            when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
            when Comp_Car_Exper_Status			= 'O' then 1 
            when DD_Conf_Ltr_Status				= 'O' then 1 
            when Comp_Fin_Mand_Status			= 'O' then 1 
            when Val_For_Spec_Item_Status		= 'O' then 1 
            when Dri_Lic_Num_Status				= 'O' then 1 
            when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
            when Orig_Cert_Status				= 'O' then 1 
            when Cancel_Req_Status				= 'O' then 1 
            when PPS_Num_Status					= 'O' then 1 
            when Afford_State_Status			= 'O' then 1 
            when Identification_Status			= 'O' then 1 
            when Digital_Journey_Status			= 'O' then 1 
            when Finance_Form_Status			= 'O' then 1 
            else 0
        End = 1 
and     Campaign = 'DAY 1'


StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [8]:
SELECT  count(distinct a.PolicyCode)
FROM    edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and     PolicyType = 'Renewals' 
and     PolicyTypeGroup = 'Motor' 
and 
        case 
            when Sign_Prop_Status				= 'O' then 1 
            when Sign_Stat_Of_Fact_Status		= 'O' then 1 
            when Comp_Fin_Agree_Status			= 'O' then 1 
            when Comp_DDM_Status				= 'O' then 1 
            when Proof_Of_NCB_Status			= 'O' then 1 
            when Trans_Proof_Of_NCB_Status		= 'O' then 1 
            when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
            when Prop_Driv_Lic_Status			= 'O' then 1 
            when Nam_Driv_Lic_Status			= 'O' then 1 
            when Gap_In_Cov_Ltr_Status			= 'O' then 1 
            when `2ndCar_Cert_Status`			= 'O' then 1 
            when Doc_Ltr_Status					= 'O' then 1 
            when Engineers_Rpt_Status			= 'O' then 1 
            when NCT_Status						= 'O' then 1 
            when Veh_Lic_Cert_Status			= 'O' then 1 
            when Irish_Reg_Status				= 'O' then 1 
            when Main_Driver_Dec_Status			= 'O' then 1 
            when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
            when Comp_Car_Exper_Status			= 'O' then 1 
            when DD_Conf_Ltr_Status				= 'O' then 1 
            when Comp_Fin_Mand_Status			= 'O' then 1 
            when Val_For_Spec_Item_Status		= 'O' then 1 
            when Dri_Lic_Num_Status				= 'O' then 1 
            when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
            when Orig_Cert_Status				= 'O' then 1 
            when Cancel_Req_Status				= 'O' then 1 
            when PPS_Num_Status					= 'O' then 1 
            when Afford_State_Status			= 'O' then 1 
            when Identification_Status			= 'O' then 1 
            when Digital_Journey_Status			= 'O' then 1 
            when Finance_Form_Status			= 'O' then 1 
            else 0
        End = 1 
and     Campaign = 'DAY 20'


StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [10]:
select  EventDescription, count(*), count(distinct TyPolicyCode) from ods.eventstream a, (select TyPolicyCode from stg.R0_MotorPoliciesEligibleForRenewals where PolicyRetNum = 1) b 
Where   SourcePolicyReference = TyPolicyCode
and     EventDateTime between  '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventSourceId = 3 
and     EVentDescription like 'Duplicate%'
Group by EventDescription
Order by 2 desc 

StatementMeta(, 22fdc5f3-46d3-43f3-98a6-e23770a088e4, 11, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 3 fields>

In [5]:
select  EventDescription, count(*), count(distinct PolicyCode) from ods.eventstream a, (select PolicyCode from stg.R0_MotorPoliciesEligibleForRenewals) b 
Where   SourcePolicyReference = PolicyCode
and     EventDateTime between  '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventSourceId = 3 
and     EventDescription like '% Held %'
and     PolicyTypeGroup = 'Motor'
and     (
            EventDescription like 'Renewal Offer - Emailed Document %'
        OR 
            EventDescription like 'Renewal Offer - Document Transmitted %'
        )
Group by EventDescription
Order by 2 desc 

StatementMeta(, f6614324-e986-4e31-854d-4b6200070f71, 6, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>

In [6]:
select  count(distinct PolicyCode) from ods.eventstream a, (select PolicyCode from stg.R0_MotorPoliciesEligibleForRenewals) b 
Where   SourcePolicyReference = PolicyCode
and     EventDateTime between  '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventSourceId = 3 
and     EventDescription like '% Held %'
and     PolicyTypeGroup = 'Motor'
and     (
            EventDescription like 'Renewal Offer - Emailed Document %'
        OR 
            EventDescription like 'Renewal Offer - Document Transmitted %'
        )


StatementMeta(, f6614324-e986-4e31-854d-4b6200070f71, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [10]:
/*
        Motor	Renewal	R2.E1
*/
select  count(distinct a.TyPolicyCode)  
from    stg.R0_MotorPoliciesEligibleForRenewals     a, 
        edw.tbl_fact_policy_renewals                b 
Where   a.PolicyCode = b.PolicyCode 
and     b.RenewalMonth = '2026-07-31'
and     RenEurFee <> TYEURFees
limit 10 

StatementMeta(, f6614324-e986-4e31-854d-4b6200070f71, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
-- get chase emails from XP table
select  count(distinct ClientCode)
from    dlk.EXT_XtremePushResults 
where   upper(campaign_name) like '%RREARS%' 
and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     campaign_name = 'Arrears - Chase 5 - 28 Days Past'
and     MessageType in ('SMS', 'EMAIL')
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Arrears - Chase 1 - 1 Day Past
*/

In [13]:
-- get chase emails from XP table
select  PolicyTypeGroup, Count(distinct b.PolicyCode)
from    dlk.EXT_XtremePushResults  a, tmp.CurrentPolicy b 
where   upper(campaign_name) like '%RREARS%' and a.ClientCode = left(b.PolicyCode,6)
and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     campaign_name = 'Arrears - Chase 1 - 1 Day Past'
and     MessageType in ('SMS', 'EMAIL')
Group by PolicyTypeGroup
order by 1 
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Arrears - Chase 1 - 1 Day Past
*/

StatementMeta(, 3cd02886-b5e0-4ec8-927b-dc2e73b8dfcb, 14, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [12]:
-- get chase emails from XP table
select  PolicyTypeGroup, Count(distinct b.PolicyCode)
from    dlk.EXT_XtremePushResults  a, tmp.CurrentPolicy b 
where   upper(campaign_name) like '%RREARS%' and a.ClientCode = left(b.PolicyCode,6)
and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     campaign_name = 'Arrears - Chase 5 - 28 Days Past'
and     MessageType in ('SMS', 'EMAIL')
Group by PolicyTypeGroup
order by 1 
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Arrears - Chase 1 - 1 Day Past
*/

StatementMeta(, 3cd02886-b5e0-4ec8-927b-dc2e73b8dfcb, 13, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [17]:
/*

    Motor	Arrears  ARR.O2

*/
select  count(distinct SourcePolicyReference) 
from            (select  distinct SourcePolicyReference 
                from    ods.eventstream        a 
                Where    EventDateTime between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
                and     EventSourceId = 3 
                and     PolicyTypeGroup = 'Motor'
                and     EventDescription like 'Insurer Led%') a,   
                (select  distinct b.PolicyCode
                from    dlk.EXT_XtremePushResults  a, tmp.CurrentPolicy b 
                where   upper(campaign_name) like '%RREARS%' and a.ClientCode = left(b.PolicyCode,6)
                and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
                --and     campaign_name = 'Arrears - Chase 5 - 28 Days Past'
                and     MessageType in ('SMS', 'EMAIL')
                and     PolicyTypeGroup = 'Motor') b 
Where   a.SourcePolicyReference = b.PolicyCode




StatementMeta(, 3cd02886-b5e0-4ec8-927b-dc2e73b8dfcb, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>